# Nathan MODE 1E - Actual Redesigned Downstream Confirmation

MODE 1D found a plausible redesign budget (minimum accepted source ring count `N = 12` within the
SLM-safe P2 radius and the current objective NA). MODE 1E now runs legitimate, fully-resolved
redesigned configurations through the **actual** inherited MODE 1 downstream machinery (ideal P2
input, vector axicon, scalar per-component focus bridge, vector ASM z-stack, F2 vectorial reference
for shortlisted candidates) and compares the sample plane against the `N = 12` source template.

This notebook does not simulate HWP/QWP/SLM physical routes, 4F carrier/iris, or panel realism,
and it cannot approve MODE 2A/2B unless the outcome is `M1E-A` from an actual downstream run.
Proxy-only, source-template, and P2 Jones passes are structurally excluded from `M1E-A`.

In [ ]:
from pathlib import Path

import vbb_study
from vbb_study.digital_twin import (
    run_mode1e_redesigned_downstream,
    write_mode1e_outputs,
)

PROJECT_ROOT = Path(vbb_study.__file__).resolve().parents[1]

In [ ]:
report = run_mode1e_redesigned_downstream(
    grid_n=384,
    z_planes=13,
    template_grid_n=384,
    template_z_planes=21,
)
report["outcome"]["suggested_outcome"], report["outcome"]["outcome_statement"]

In [ ]:
# Requested vs resolved design numbers for every candidate (plus the inherited control row).
list(report["candidate_rows"])

In [ ]:
# The N=12 primary template and the stricter N=31 reference template.
{key: {
    "classified": tmpl.symmetry_class,
    "accepted": tmpl.accepted_hexagon,
    "ring_radius_um": tmpl.ring_radius_m / 1e-6,
    "actual_ring_count": tmpl.ring_count_actual,
} for key, tmpl in report["templates"].items()}

In [ ]:
# Control: the unmodified inherited geometry must remain non-hexagonal.
{
    "control_class": report["control"].candidate.symmetry_class,
    "control_gate_pass": report["control"].template_gate_pass,
    "shortlist_f2_candidate_ids": report["shortlist_candidate_ids"],
    "mode2a_2b_gate": report["outcome"]["mode2a_2b_gate"],
}

In [ ]:
paths = write_mode1e_outputs(
    output_dir=PROJECT_ROOT / "outputs" / "figures" / "digital_twin" / "nathan_mode1e_redesigned_downstream",
    report=report,
)
{key: str(value) for key, value in paths.items()}

Expected outcome: the design budget is honoured (`N = 12` at the SLM-safe radius needs
`NA ~ 0.39 < 0.45`) and the redesigned configs resolve their requested pre/surface `k_r` exactly,
so the run is a legitimate actual-downstream test, not a proxy. The final `M1E-A/B/C/D` outcome and
the MODE 2A/2B gate state are read from `mode1e_outcome_report.json`; MODE 2A/2B remains blocked
unless the outcome is `M1E-A`.